# Teste do GuruPass RAG — GymSite

**Grupo:** `4d1e2c40-217b-4a39-bc08-f9c3e90fd803` (GuruPass Brasil)
**Status:** ~5008 chunks `gym_modality` · `municipios_relacionados` preenchido
**Agent:** published

## O que este notebook testa
1. **Filtro de cidade** via `match_municipio` ↔ `meta.municipios_relacionados` (ex-`municipios_busca`)
2. **Modalidade + créditos** (musculação, yoga, boxe, jiu-jitsu, pilates)
3. **Recuperação por academia esperada** (`source_ref` / `gym_id`)
4. **Métricas** Recall@5, Precision@5, MRR, **hit_rate**, city_match (+ document_match só diagnóstico)

Busca = embed Ollama (`mxbai-embed-large` @ 1024) + RPC `match_chunks` (+ `match_municipio`).

## Requisitos
```bash
# use o kernel Python 3.13 (.venv assistent-control)
```

**Importante:** Kernel → Restart & Run All. Cwd costuma ser `notebooks/` — setup resolve ROOT.


In [1]:
# Configuracao — rode ESTA cell antes de tudo
import os
import json
from pathlib import Path
from typing import Any, Dict, List, cast
from collections import Counter
from dotenv import load_dotenv
from supabase import create_client, Client

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

for env_name in (".env", ".env.local"):
    env_path = ROOT / env_name
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        print(f"loaded {env_path}")

SUPABASE_URL = os.getenv("SUPABASE_URL") or os.getenv("VITE_SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
GURUPASS_GROUP_ID = (
    os.getenv("GURUPASS_GROUP_ID") or "4d1e2c40-217b-4a39-bc08-f9c3e90fd803"
)

OLLAMA_BASE = (
    os.getenv("OLLAMA_BASE_URL") or "https://ollama2.vectracargo.com.br"
).rstrip("/").removesuffix("/v1")
EMBED_MODEL = os.getenv("EMBEDDING_MODEL") or "mxbai-embed-large"
EMBED_DIM = int(os.getenv("EMBEDDING_DIMENSION") or "1024")
MIN_SIM = float(os.getenv("RAG_MIN_SIMILARITY") or "0.35")
TOP_K = int(os.getenv("RAG_TOP_K") or "5")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(
        f"SUPABASE_URL / SUPABASE_SERVICE_ROLE_KEY ausentes (ROOT={ROOT})"
    )

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"Supabase: {SUPABASE_URL}")
print(f"GuruPass group: {GURUPASS_GROUP_ID}")
print(f"Embed: {EMBED_MODEL} @ {OLLAMA_BASE} dim={EMBED_DIM} min_sim={MIN_SIM} top_k={TOP_K}")

agent = (
    supabase.table("eros_knowledge_agents")
    .select("*")
    .eq("group_id", GURUPASS_GROUP_ID)
    .execute()
)
if agent.data:
    a = cast(Dict[str, Any], agent.data[0])
    print(f"Agente: {a['name']} · status={a['status']} · chunks={a['chunk_count']}")
else:
    print("Agente nao encontrado")

types = (
    supabase.table("eros_knowledge_chunks")
    .select("chunk_type, embedding_model")
    .eq("group_id", GURUPASS_GROUP_ID)
    .execute()
)
rows = cast(List[Dict[str, Any]], types.data or [])
pending = sum(1 for r in rows if (r.get("embedding_model") or "") == "pending")
print("Tipos:", dict(Counter((r.get("chunk_type") or "?") for r in rows)))
print(f"Sample rows={len(rows)} pending={pending}")


loaded C:\Users\marce\assistent-control\.env.local
Supabase: https://gxmaxbjgdrqdcizvdojp.supabase.co
GuruPass group: 4d1e2c40-217b-4a39-bc08-f9c3e90fd803
Embed: mxbai-embed-large @ https://ollama2.vectracargo.com.br dim=1024 min_sim=0.35 top_k=5
Agente: GuruPass Brasil · status=published · chunks=5008
Tipos: {'gym_modality': 1000}
Sample rows=1000 pending=0


In [2]:
# Embeddings + match_chunks (+ match_municipio)
from typing import Any, Dict, List, Optional, cast
import httpx
import unicodedata

def _norm(s: str) -> str:
    s = (s or "").lower()
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )

def embed_query(text: str) -> List[float]:
    url = f"{OLLAMA_BASE}/v1/embeddings"
    with httpx.Client(timeout=60.0) as client:
        r = client.post(url, json={"model": EMBED_MODEL, "input": text[:1000]})
        r.raise_for_status()
        data = r.json()
    vec = data["data"][0]["embedding"]
    if len(vec) != EMBED_DIM:
        raise ValueError(f"dim mismatch: got {len(vec)} expected {EMBED_DIM}")
    return vec

def search_chunks(
    query: str,
    top_k: Optional[int] = None,
    min_similarity: Optional[float] = None,
    municipio: Optional[str] = None,
    modalidade: Optional[str] = None,
) -> List[Dict[str, Any]]:
    embedding = embed_query(query)
    result = supabase.rpc(
        "match_chunks",
        {
            "query_embedding": embedding,
            "match_group_id": GURUPASS_GROUP_ID,
            "match_tenant_id": None,
            "match_modalidade": modalidade,
            "match_bairro": None,
            "match_plano_rank": None,
            "match_municipio": municipio,
            "match_k": top_k if top_k is not None else TOP_K,
            "min_similarity": min_similarity if min_similarity is not None else MIN_SIM,
            "match_query": query,
        },
    ).execute()
    return cast(List[Dict[str, Any]], result.data or [])

print("Funcoes OK — search = embed + match_chunks (+ match_municipio / match_modalidade)")


Funcoes OK — search = embed + match_chunks (+ match_municipio / match_modalidade)


## Dataset de Teste — GuruPass

Queries com **cidade explícita** exercitam `match_municipio` contra `meta.municipios_relacionados`.
`expected_doc` = `gym_id` / `source_ref` da academia âncora.


In [3]:
# Dataset de avaliacao — GuruPass (cidade + modalidade + créditos)
EVAL_DATASET = [
    {
        "id": "gp-01",
        "query": "Academia de musculação em Arujá no GuruPass a partir de quantos créditos?",
        "expected_keywords": ["Arujá", "musculação", "créditos"],
        "expected_doc": "b9f7b844-881f-49db-a895-c02d366cd96e",  # Studio Xtreme
        "expected_cidade": "Arujá",
        "expected_uf": "SP",
    },
    {
        "id": "gp-02",
        "query": "Onde fazer musculação em São Paulo com GuruPass barato em créditos?",
        "expected_keywords": ["São Paulo", "musculação", "créditos"],
        "expected_doc": "23443b34-629e-410a-b043-b9eb7c2107f3",  # Evoque Rio Branco
        "expected_cidade": "São Paulo",
        "expected_uf": "SP",
    },
    {
        "id": "gp-03",
        "query": "Academia de boxe em Osasco que aceita GuruPass",
        "expected_keywords": ["Osasco", "boxe", "GuruPass"],
        "expected_doc": "ea5dbc61-ad1c-442d-a9f5-025cc7752d2b",  # BrasCuba
        "expected_cidade": "Osasco",
        "expected_uf": "SP",
    },
    {
        "id": "gp-04",
        "query": "Jiu-jitsu em Guarulhos no GuruPass quantos créditos?",
        "expected_keywords": ["Guarulhos", "jiu", "créditos"],
        "expected_doc": "24c67f72-c8e2-4975-ad39-e81b6db83502",  # Cabapuã
        "expected_cidade": "Guarulhos",
        "expected_uf": "SP",
    },
    {
        "id": "gp-05",
        "query": "Musculação em Curitiba via GuruPass Studio Happiness",
        "expected_keywords": ["Curitiba", "musculação", "Happiness"],
        "expected_doc": "defd6c33-dba4-4ab5-933a-aa2d73fbd0f6",
        "expected_cidade": "Curitiba",
        "expected_uf": "PR",
    },
    {
        "id": "gp-06",
        "query": "Yoga em Campinas no GuruPass",
        "expected_keywords": ["Campinas", "yoga", "créditos"],
        "expected_doc": "e49a473a-32c9-4014-bdc2-433afd744930",  # Max Premium
        "expected_cidade": "Campinas",
        "expected_uf": "SP",
    },
    {
        "id": "gp-07",
        "query": "Pilates em Niterói com GuruPass",
        "expected_keywords": ["Niterói", "pilates", "créditos"],
        "expected_doc": "d6bccd92-3ed5-47cc-bce4-e6d8367c25b7",
        "expected_cidade": "Niterói",
        "expected_uf": "RJ",
    },
    {
        "id": "gp-08",
        "query": "Musculação em Santos ou São Vicente no GuruPass",
        "expected_keywords": ["Santos", "musculação", "créditos"],
        "expected_doc": "549fa757-31e6-487d-a4a1-79814d8f6076",
        "expected_cidade": "Santos",
        "expected_uf": "SP",
    },
]

print(f"Dataset de avaliacao: {len(EVAL_DATASET)} queries")
print(f"Academias esperadas: {len(set(q['expected_doc'] for q in EVAL_DATASET))}")
print(f"Cidades com city_match: {sum(1 for q in EVAL_DATASET if q.get('expected_cidade'))}")


Dataset de avaliacao: 8 queries
Academias esperadas: 8
Cidades com city_match: 8


## Funções de avaliação

- `city_match` — ancora em `meta.cidade` + `meta.uf` (rejeita falso positivo tipo `Campinas / RS`). `municipios_relacionados` é fallback de região, não verdade geográfica.
- `hit_rate` — métrica certa p/ domínio de **conjunto**: ≥1 dos top-5 atende cidade ∧ modalidade. Preferir sobre `document_match` (âncora única = inadequada aqui).
- `document_match` — mantido só como diagnóstico de gym_id âncora (não gate de qualidade).


In [4]:
def _parse_meta(chunk: Dict[str, Any]) -> Dict[str, Any]:
    meta = chunk.get("meta") or {}
    if isinstance(meta, str):
        try:
            meta = json.loads(meta)
        except Exception:
            meta = {}
    return meta if isinstance(meta, dict) else {}

def chunk_blob(chunk: Dict[str, Any]) -> str:
    return _norm(
        " ".join(
            [
                str(chunk.get("text") or ""),
                str(chunk.get("meta") or ""),
                str(chunk.get("source_ref") or ""),
            ]
        )
    )

def chunk_is_relevant(chunk: Dict[str, Any], expected_keywords: List[str]) -> bool:
    blob = chunk_blob(chunk)
    return any(_norm(kw) in blob for kw in expected_keywords)

def calculate_recall_at_k(retrieved, expected_keywords, k=5) -> float:
    if not expected_keywords:
        return 1.0
    blob = " ".join(chunk_blob(c) for c in retrieved[:k])
    hit = sum(1 for kw in expected_keywords if _norm(kw) in blob)
    return hit / len(expected_keywords)

def calculate_precision_at_k(retrieved, expected_keywords, k=5) -> float:
    if not retrieved[:k]:
        return 0.0
    relevant = sum(1 for c in retrieved[:k] if chunk_is_relevant(c, expected_keywords))
    return relevant / min(k, len(retrieved[:k]))

def calculate_mrr(retrieved, expected_keywords) -> float:
    for i, chunk in enumerate(retrieved):
        if chunk_is_relevant(chunk, expected_keywords):
            return 1.0 / (i + 1)
    return 0.0

def _chunk_source_refs(chunk: Dict[str, Any]) -> List[str]:
    refs: List[str] = []
    top = chunk.get("source_ref")
    if top:
        refs.append(str(top))
    meta = _parse_meta(chunk)
    for key in ("source_ref", "gym_id", "nome_academia"):
        val = meta.get(key)
        if val:
            refs.append(str(val))
    return refs

def check_document_match(retrieved: List[Dict[str, Any]], expected_doc: str) -> bool:
    """Âncora gym_id única — diagnóstico; NÃO use como gate no domínio de conjunto."""
    needle = _norm(expected_doc)
    for chunk in retrieved:
        for ref in _chunk_source_refs(chunk):
            if needle in _norm(ref) or _norm(ref) in needle:
                return True
    return False

def _parse_cidade_uf(cidade_raw: str) -> tuple[str, Optional[str]]:
    """'Campinas / RS' → ('Campinas', 'RS'); 'São Paulo' → ('São Paulo', None)."""
    raw = str(cidade_raw or "").strip()
    if "/" in raw:
        left, right = raw.split("/", 1)
        maybe_uf = right.strip().upper()
        if len(maybe_uf) == 2 and maybe_uf.isalpha():
            return left.strip(), maybe_uf
    return raw, None

def _city_uf_ok(
    meta: Dict[str, Any],
    expected_cidade: str,
    expected_uf: Optional[str] = None,
) -> bool:
    """Ancora cidade (+ UF quando disponível). Rejeita Campinas/RS vs Campinas/SP."""
    needle = _norm(expected_cidade)
    cidade_raw = str(meta.get("cidade") or "")
    cidade_nome, cidade_uf_embedded = _parse_cidade_uf(cidade_raw)
    meta_uf = (str(meta.get("uf") or "").strip().upper() or None)
    effective_uf = meta_uf or cidade_uf_embedded

    if needle not in _norm(cidade_nome) and needle not in _norm(cidade_raw):
        return False
    if expected_uf and effective_uf and effective_uf != expected_uf.upper():
        return False
    return True

def check_city_match(
    retrieved: List[Dict[str, Any]],
    expected_cidade: Optional[str],
    expected_uf: Optional[str] = None,
) -> Optional[bool]:
    if not expected_cidade:
        return None
    needle = _norm(expected_cidade)
    for chunk in retrieved:
        meta = _parse_meta(chunk)
        if _city_uf_ok(meta, expected_cidade, expected_uf):
            return True
        # Fallback região: só se UF bate (ou UF ausente no expected)
        munis = meta.get("municipios_relacionados") or []
        if isinstance(munis, list):
            meta_uf = (str(meta.get("uf") or "").strip().upper() or None)
            if expected_uf and meta_uf and meta_uf != expected_uf.upper():
                continue
            for m in munis:
                if needle in _norm(str(m)):
                    return True
    return False

def check_hit_rate(
    retrieved: List[Dict[str, Any]],
    expected_keywords: List[str],
    expected_cidade: Optional[str],
    expected_uf: Optional[str] = None,
    k: int = 5,
) -> bool:
    """≥1 dos top-k atende modalidade (keywords) ∧ cidade(+uf). Métrica correta p/ conjunto."""
    for chunk in retrieved[:k]:
        if not chunk_is_relevant(chunk, expected_keywords):
            continue
        if not expected_cidade:
            return True
        if _city_uf_ok(_parse_meta(chunk), expected_cidade, expected_uf):
            return True
    return False

def evaluate_query(
    query: str,
    expected_keywords: List[str],
    expected_doc: str,
    expected_cidade: Optional[str] = None,
    expected_uf: Optional[str] = None,
    top_k: Optional[int] = None,
    min_similarity: Optional[float] = None,
) -> Dict[str, Any]:
    retrieved = search_chunks(
        query,
        top_k=top_k,
        min_similarity=min_similarity,
        municipio=expected_cidade,
    )
    city = check_city_match(retrieved, expected_cidade, expected_uf)
    hit = check_hit_rate(
        retrieved, expected_keywords, expected_cidade, expected_uf, k=5
    )
    return {
        "query": query,
        "expected_doc": expected_doc,
        "expected_cidade": expected_cidade,
        "expected_uf": expected_uf,
        "retrieved_count": len(retrieved),
        "recall@5": calculate_recall_at_k(retrieved, expected_keywords, k=5),
        "precision@5": calculate_precision_at_k(retrieved, expected_keywords, k=5),
        "mrr": calculate_mrr(retrieved, expected_keywords),
        "hit_rate": hit,
        "document_match": check_document_match(retrieved, expected_doc),
        "city_match": city,
        "top_sims": [round(float(c.get("similarity") or 0), 3) for c in retrieved[:3]],
        "chunks": retrieved[:3],
    }

print("Funcoes de avaliacao OK (hit_rate + cidade/uf)")


Funcoes de avaliacao OK (hit_rate + cidade/uf)


## Smoke — filtro cidade

Compara a mesma query **com** e **sem** `match_municipio=Arujá`.


In [5]:
# Smoke: match_municipio liga/desliga
q = "musculação GuruPass créditos"
plain = search_chunks(q, municipio=None)
filtered = search_chunks(q, municipio="Arujá")

def _cities(chunks):
    out = []
    for c in chunks[:5]:
        m = _parse_meta(c)
        out.append({
            "cidade": m.get("cidade"),
            "mun": (m.get("municipios_relacionados") or [])[:3],
            "nome": m.get("nome_academia"),
            "sim": round(float(c.get("similarity") or 0), 3),
        })
    return out

print("SEM municipio:")
for row in _cities(plain):
    print(" ", row)
print("\nCOM match_municipio=Arujá:")
for row in _cities(filtered):
    print(" ", row)


SEM municipio:
  {'cidade': 'Campo Grande', 'mun': ['Campo Grande', 'Cuiabá', 'Porto Velho'], 'nome': 'Studio Energy Treinamento Personalizado de Musculação', 'sim': 0.865}
  {'cidade': 'Campo Grande', 'mun': ['Campo Grande', 'Cuiabá', 'Porto Velho'], 'nome': 'Studio Energy Treinamento Personalizado de Musculação', 'sim': 0.853}
  {'cidade': 'Blumenau', 'mun': ['Joinville', 'Florianópolis', 'Blumenau'], 'nome': 'UX Multifit', 'sim': 0.849}
  {'cidade': 'Ouroeste', 'mun': ['Cuiabá', 'Várzea Grande', 'Rondonópolis'], 'nome': 'Academia Delta Fitness', 'sim': 0.845}
  {'cidade': 'Americana', 'mun': ['Limeira', 'Sumaré', 'Americana'], 'nome': 'On fit Studio Treinamento Personalizado', 'sim': 0.842}

COM match_municipio=Arujá:
  {'cidade': 'Guarulhos', 'mun': ['Arujá', 'Santa Isabel', 'Piracaia'], 'nome': 'Fisiko Fit Academia LTDA', 'sim': 0.834}
  {'cidade': 'Cotia', 'mun': ['Arujá', 'Santa Isabel', 'Piracaia'], 'nome': 'NewFit Natação e Musculação - Unidade 2', 'sim': 0.824}
  {'cidade': '

## Executar Avaliação

Para cada query: embed → `match_chunks` com `match_municipio=expected_cidade` → métricas.


In [6]:
# Executar avaliacao completa
results = []

print("Executando avaliacao GuruPass (vector + match_municipio)...\n")
print("=" * 80)

for eval_item in EVAL_DATASET:
    print(f"\n[{eval_item['id']}] {eval_item['query']}")
    print(f"   Esperado: {eval_item['expected_doc']} · cidade={eval_item.get('expected_cidade')}")
    print(f"   Keywords: {eval_item['expected_keywords']}")

    result = evaluate_query(
        query=eval_item["query"],
        expected_keywords=eval_item["expected_keywords"],
        expected_doc=eval_item["expected_doc"],
        expected_cidade=eval_item.get("expected_cidade"),
        expected_uf=eval_item.get("expected_uf"),
        top_k=TOP_K,
        min_similarity=MIN_SIM,
    )
    result["id"] = eval_item["id"]
    results.append(result)

    city_s = (
        "N/A" if result["city_match"] is None else ("YES" if result["city_match"] else "NO")
    )
    print(f"   Recuperados: {result['retrieved_count']} chunks sims={result['top_sims']}")
    print(f"   Recall@5: {result['recall@5']:.2f}")
    print(f"   Precision@5: {result['precision@5']:.2f}")
    print(f"   MRR: {result['mrr']:.2f}")
    print(f"   document_match: {'YES' if result['document_match'] else 'NO'}")
    print(f"   hit_rate: {'YES' if result['hit_rate'] else 'NO'}")
    print(f"   city_match: {city_s}")
    print(f"   document_match (âncora): {'YES' if result['document_match'] else 'NO'}")

    if result["chunks"]:
        first = result["chunks"][0]
        refs = _chunk_source_refs(first)
        meta = _parse_meta(first)
        preview = (first.get("text") or "")[:120].replace("\n", " ")
        print(f"   Primeiro: {refs[0] if refs else 'N/A'} · cidade={meta.get('cidade')}")
        print(f"   Preview: {preview}...")

print("\n" + "=" * 80)


Executando avaliacao GuruPass (vector + match_municipio)...


[gp-01] Academia de musculação em Arujá no GuruPass a partir de quantos créditos?
   Esperado: b9f7b844-881f-49db-a895-c02d366cd96e · cidade=Arujá
   Keywords: ['Arujá', 'musculação', 'créditos']
   Recuperados: 5 chunks sims=[0.879, 0.863, 0.871]
   Recall@5: 1.00
   Precision@5: 1.00
   MRR: 1.00
   document_match: NO
   hit_rate: YES
   city_match: YES
   document_match (âncora): NO
   Primeiro: 1c9da7fd-1c5a-4b7b-950e-46adc9d8553a · cidade=Guarulhos
   Preview: Academia: Fisiko Fit Academia LTDA Cidade: Guarulhos / SP Bairro: Residencial e Comercial Guarulhos Endereço: Rua Um, Re...

[gp-02] Onde fazer musculação em São Paulo com GuruPass barato em créditos?
   Esperado: 23443b34-629e-410a-b043-b9eb7c2107f3 · cidade=São Paulo
   Keywords: ['São Paulo', 'musculação', 'créditos']
   Recuperados: 5 chunks sims=[0.863, 0.861, 0.849]
   Recall@5: 1.00
   Precision@5: 1.00
   MRR: 1.00
   document_match: NO
   hit_rate: YES
  

## Relatório de Avaliação


In [7]:
if not results:
    raise RuntimeError("results vazio — rode a cell de avaliacao antes")

avg_recall = sum(r["recall@5"] for r in results) / len(results)
avg_precision = sum(r["precision@5"] for r in results) / len(results)
avg_mrr = sum(r["mrr"] for r in results) / len(results)
hit_rate = sum(1 for r in results if r.get("hit_rate")) / len(results)
doc_match_rate = sum(1 for r in results if r["document_match"]) / len(results)  # diagnóstico
city_scored = [r for r in results if r["city_match"] is not None]
city_match_rate = (
    sum(1 for r in city_scored if r["city_match"]) / len(city_scored)
    if city_scored
    else None
)

print("Relatorio de Avaliacao — GuruPass")
print("=" * 50)
print(f"Total de queries: {len(results)}")
print(f"Recall@5 medio: {avg_recall:.2f}")
print(f"Precision@5 medio: {avg_precision:.2f}")
print(f"MRR medio: {avg_mrr:.2f}")
print(f"Taxa hit_rate: {hit_rate:.0%}  ← métrica de conjunto")
print(f"Taxa document_match (âncora): {doc_match_rate:.0%}  ← diagnóstico, não gate")
if city_match_rate is not None:
    print(f"Taxa city_match: {city_match_rate:.0%} ({len(city_scored)} queries)")
print("=" * 50)

print("\nQueries com Recall@5 < 0.5:")
low_recall = [r for r in results if r["recall@5"] < 0.5]
if low_recall:
    for r in low_recall:
        print(f"  - {r['query'][:60]}...: Recall@5={r['recall@5']:.2f}")
else:
    print("  Nenhuma")

print("\nQueries sem hit_rate (cidade∧modalidade):")
no_hit = [r for r in results if not r.get("hit_rate")]
if no_hit:
    for r in no_hit:
        print(f"  - {r['query'][:60]}...")
else:
    print("  Todas com ≥1 resultado válido no top-5")

print("\nQueries sem academia-âncora (document_match — diagnóstico):")
no_doc = [r for r in results if not r["document_match"]]
if no_doc:
    for r in no_doc:
        print(f"  - {r['query'][:60]}...: esperado {r['expected_doc']}")
else:
    print("  Todas as academias esperadas foram recuperadas")

print("\nQueries sem cidade esperada:")
no_city = [r for r in city_scored if not r["city_match"]]
if no_city:
    for r in no_city:
        print(f"  - {r['query'][:60]}...: esperado {r['expected_cidade']}")
elif city_scored:
    print("  Todas as cidades esperadas foram recuperadas")
else:
    print("  N/A")


Relatorio de Avaliacao — GuruPass
Total de queries: 8
Recall@5 medio: 1.00
Precision@5 medio: 1.00
MRR medio: 1.00
Taxa hit_rate: 100%  ← métrica de conjunto
Taxa document_match (âncora): 38%  ← diagnóstico, não gate
Taxa city_match: 100% (8 queries)

Queries com Recall@5 < 0.5:
  Nenhuma

Queries sem hit_rate (cidade∧modalidade):
  Todas com ≥1 resultado válido no top-5

Queries sem academia-âncora (document_match — diagnóstico):
  - Academia de musculação em Arujá no GuruPass a partir de quan...: esperado b9f7b844-881f-49db-a895-c02d366cd96e
  - Onde fazer musculação em São Paulo com GuruPass barato em cr...: esperado 23443b34-629e-410a-b043-b9eb7c2107f3
  - Jiu-jitsu em Guarulhos no GuruPass quantos créditos?...: esperado 24c67f72-c8e2-4975-ad39-e81b6db83502
  - Yoga em Campinas no GuruPass...: esperado e49a473a-32c9-4014-bdc2-433afd744930
  - Musculação em Santos ou São Vicente no GuruPass...: esperado 549fa757-31e6-487d-a4a1-79814d8f6076

Queries sem cidade esperada:
  Todas as ci

## Análise por cidade / academia


In [8]:
doc_counts: Dict[str, int] = {}
city_counts: Dict[str, int] = {}
for r in results:
    for chunk in r["chunks"]:
        refs = _chunk_source_refs(chunk)
        key = refs[0] if refs else "N/A"
        doc_counts[key] = doc_counts.get(key, 0) + 1
        meta = _parse_meta(chunk)
        cidade = meta.get("cidade")
        if cidade:
            city_counts[str(cidade)] = city_counts.get(str(cidade), 0) + 1

print("Academias recuperadas (top 8):")
for doc, count in sorted(doc_counts.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f"  {doc}: {count} vezes")

print("\nCidades recuperadas (meta.cidade):")
for cidade, count in sorted(city_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cidade}: {count} vezes")

print("\nDistribuicao por academia esperada:")
for doc in sorted(set(r["expected_doc"] for r in results)):
    qs = [r for r in results if r["expected_doc"] == doc]
    avg_r = sum(r["recall@5"] for r in qs) / len(qs)
    match_rate = sum(1 for r in qs if r["document_match"]) / len(qs)
    print(
        f"  {doc}: {len(qs)} queries · "
        f"Recall@5={avg_r:.2f} · doc_match={match_rate:.0%}"
    )


Academias recuperadas (top 8):
  defd6c33-dba4-4ab5-933a-aa2d73fbd0f6: 3 vezes
  1c9da7fd-1c5a-4b7b-950e-46adc9d8553a: 1 vezes
  cf5c3f24-c45b-45d0-88d1-5b24511cd9d7: 1 vezes
  15823912-6275-4ab3-aa50-13a9ea6883e2: 1 vezes
  118112e5-376e-4ffb-bcf7-b3de14f67c4d: 1 vezes
  d1df15ce-ed98-49f7-9cb1-7241834231a7: 1 vezes
  7a60ca87-105b-41e2-b02a-28ba75690e47: 1 vezes
  ea5dbc61-ad1c-442d-a9f5-025cc7752d2b: 1 vezes

Cidades recuperadas (meta.cidade):
  Guarulhos: 4 vezes
  São Paulo: 3 vezes
  Osasco: 3 vezes
  Curitiba: 3 vezes
  Campinas: 3 vezes
  Niterói: 3 vezes
  Santos: 3 vezes
  Cotia: 1 vezes
  Poá: 1 vezes

Distribuicao por academia esperada:
  23443b34-629e-410a-b043-b9eb7c2107f3: 1 queries · Recall@5=1.00 · doc_match=0%
  24c67f72-c8e2-4975-ad39-e81b6db83502: 1 queries · Recall@5=1.00 · doc_match=0%
  549fa757-31e6-487d-a4a1-79814d8f6076: 1 queries · Recall@5=1.00 · doc_match=0%
  b9f7b844-881f-49db-a895-c02d366cd96e: 1 queries · Recall@5=1.00 · doc_match=0%
  d6bccd92-3ed5-47c

## Salvar Resultados

Salva em `data/evaluation/gurupass_eval_results.json`.


In [9]:
from datetime import datetime

output_path = ROOT / "data" / "evaluation" / "gurupass_eval_results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

serializable_results = []
for r in results:
    row = {k: v for k, v in r.items() if k != "chunks"}
    row["chunks_preview"] = [
        {
            "source_ref": (_chunk_source_refs(c) or ["N/A"])[0],
            "cidade": _parse_meta(c).get("cidade"),
            "municipios_relacionados": (_parse_meta(c).get("municipios_relacionados") or [])[:5],
            "nome_academia": _parse_meta(c).get("nome_academia"),
            "similarity": c.get("similarity"),
            "text": (c.get("text") or "")[:240],
        }
        for c in r.get("chunks") or []
    ]
    serializable_results.append(row)

eval_report = {
    "timestamp": datetime.now().isoformat(),
    "group_id": GURUPASS_GROUP_ID,
    "embed_model": EMBED_MODEL,
    "embed_dim": EMBED_DIM,
    "min_similarity": MIN_SIM,
    "top_k": TOP_K,
    "metrics": {
        "avg_recall@5": avg_recall,
        "avg_precision@5": avg_precision,
        "avg_mrr": avg_mrr,
        "hit_rate": hit_rate,
        "doc_match_rate": doc_match_rate,  # diagnóstico
        "city_match_rate": city_match_rate,
    },
    "results": serializable_results,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(eval_report, f, indent=2, ensure_ascii=False)

print(f"Resultados salvos em: {output_path}")


Resultados salvos em: C:\Users\marce\assistent-control\data\evaluation\gurupass_eval_results.json


##### Conclusão

Grupo GuruPass: **~5008** chunks com `municipios_relacionados`.
Eval foca `city_match` — prova que `match_municipio` casa com o scrape enriquecido.
Rode **Restart & Run All** e confira `data/evaluation/gurupass_eval_results.json`.
